
# 03 | Source Data Structure and Integration Readiness

## Project
Victorian Local Government Financial Sustainability
& Performance Intelligence (2020–2025)

## Purpose
Understand the structure of the council performance,
financial and population datasets before combining them.

## Objectives
1. Compare column names across source files.
2. Identify council and financial-year fields.
3. Investigate unusual or missing headers.
4. Understand the level of detail in each dataset.
5. Identify potential integration problems.

## Previous Work
- Notebook 01: Bronze ingestion completed.
- Notebook 02: Data quality assessment completed.

## Expected Outcome
A documented source schema assessment that helps
prepare reliable financial and population datasets.

No source data will be modified in this notebook.


In [0]:

from pyspark.sql import functions as F
import re

base = "/Volumes/workspace/default/council_raw"

files = [f for f in dbutils.fs.ls(base) if f.name.lower().endswith(".csv")]

rows = []

for file in files:
    data = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(file.path)
    )

    for col in data.columns:
        normalised = re.sub(r"[^a-z0-9]", "", col.lower())
        rows.append((file.name, col, normalised))

headers = spark.createDataFrame(
    rows, ["File", "OriginalColumn", "NormalisedName"]
)

display(headers.orderBy("NormalisedName", "File"))

# Identify columns with formatting differences
matches = (
    headers.groupBy("NormalisedName")
    .agg(
        F.countDistinct("OriginalColumn").alias("NameVariants"),
        F.collect_set("OriginalColumn").alias("ColumnNames"),
        F.collect_set("File").alias("Files")
    )
    .filter("NameVariants > 1")
)

display(matches)


File,OriginalColumn,NormalisedName
Indicator data.csv,Attribute,attribute
Per-capita financial data.csv,Attribute,attribute
Estimated resident population Local Government Areas Australia.csv,_c1,c1
Estimated resident population Local Government Areas Australia.csv,_c10,c10
Estimated resident population Local Government Areas Australia.csv,_c11,c11
Estimated resident population Local Government Areas Australia.csv,_c12,c12
Estimated resident population Local Government Areas Australia.csv,_c13,c13
Estimated resident population Local Government Areas Australia.csv,_c14,c14
Estimated resident population Local Government Areas Australia.csv,_c15,c15
Estimated resident population Local Government Areas Australia.csv,_c16,c16


NormalisedName,NameVariants,ColumnNames,Files


In [0]:

from pyspark.sql import functions as F

raw_folder = "/Volumes/workspace/default/council_raw/"

# List all CSV files
files = [
    f.name for f in dbutils.fs.ls(raw_folder)
    if f.name.lower().endswith(".csv")
]

schema_rows = []

# Read each file and collect its column names
for filename in sorted(files):
    df_source = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(raw_folder + filename)
    )

    schema_rows.append((
        filename,
        len(df_source.columns),
        ", ".join(df_source.columns)
    ))

# Create the schema summary
schema_summary = spark.createDataFrame(
    schema_rows,
    ["File", "Total_Columns", "Column_Names"]
)

display(schema_summary)


File,Total_Columns,Column_Names
Council category data.csv,2,"Council, Council Category"
Estimated resident population Local Government Areas Australia.csv,37,"This tab has one table with the estimated resident population for Local Government Areas in Australia as at 30 June 2001 to 30 June 2025. It ranges from cell A1 to AA559., _c1, _c2, _c3, _c4, _c5, _c6, _c7, _c8, _c9, _c10, _c11, _c12, _c13, _c14, _c15, _c16, _c17, _c18, _c19, _c20, _c21, _c22, _c23, _c24, _c25, _c26, _c27, _c28, _c29, _c30, _c31, _c32, _c33, _c34, _c35, _c36"
Indicator data.csv,4,"Council, Year, Attribute, Value"
Indicators 2020-21.csv,8,"Council, Group, ID, Description, Service Provided, Data applicable?, Result, Variation"
Indicators 2021-22.csv,8,"Council, Group, ID, Description, Service Provided, Data applicable?, Result, Variation"
Indicators 2022-23.csv,8,"Council, Group, ID, Description, Service Provided, Data applicable?, Result, Variation"
Indicators 2023-24.csv,9,"Council, Group, ID, Description, Service Provided, Data applicable?, Result, Target, Variation"
Indicators 2024-25.csv,9,"Council, Group, ID, Description, Service Provided, Data applicable?, Result, Target, Variation"
Per-capita financial data.csv,4,"Council, Year, Attribute, Value"
Population.csv,5,"Council, Year, Population*, _c3, *Source: Australian Bureau of Statistics."


In [0]:

# Inspect the raw population CSV
population_path = (
    "/Volumes/workspace/default/council_raw/"
    "Estimated resident population Local Government Areas Australia.csv"
)

population_raw = (
    spark.read
    .option("header", False)
    .csv(population_path)
)

print("Population columns:", len(population_raw.columns))

# Show the first 15 rows to locate the actual headers
display(population_raw.limit(15))


Population columns: 37


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18,_c19,_c20,_c21,_c22,_c23,_c24,_c25,_c26,_c27,_c28,_c29,_c30,_c31,_c32,_c33,_c34,_c35,_c36
This tab has one table with the estimated resident population for Local Government Areas in Australia as at 30 June 2001 to 30 June 2025. It ranges from cell A1 to AA559.,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"Table 1. Estimated resident population, Local Government Areas (a), Australia",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"Regional population, 2024–25",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,ERP at 30 June (b),null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,null,null,null,null,null,null,null,null,null,null
LGA code,LGA name,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,no.,null,null,null,null,null,null,null,null,null,null
10050,Albury,45265,45816,46180,46505,47004,47566,48140,48518,48814,49222,49451,49785,50338,50990,51486,52171,53056,53922,54657,55466,56067,57024,57880,58768,59538,null,null,null,null,null,null,null,null,null,null
10180,Armidale,27906,27774,27610,27410,27350,27377,27468,27788,28156,28409,28573,28748,28895,29015,29160,29310,29519,29631,29701,29600,29332,29433,29639,29686,29652,null,null,null,null,null,null,null,null,null,null
10250,Ballina,37856,38417,38870,39120,39305,39537,39824,40020,40295,40571,40747,41118,41510,41881,42336,42993,43652,44385,44997,45663,46196,47002,47423,48135,48603,null,null,null,null,null,null,null,null,null,null
10300,Balranald,2751,2703,2661,2596,2545,2507,2473,2433,2404,2381,2353,2363,2381,2376,2364,2330,2338,2308,2287,2257,2208,2236,2224,2256,2244,null,null,null,null,null,null,null,null,null,null


In [0]:

from pyspark.sql import functions as F

population = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/council_raw/Population.csv")
)

print("Columns:", population.columns)
print("Total rows:", population.count())

display(population.limit(10))

display(
    population.groupBy("Year")
    .agg(
        F.count("*").alias("Rows"),
        F.countDistinct("Council").alias("Councils")
    )
    .orderBy("Year")
)


Columns: ['Council', 'Year', 'Population*', '_c3', '*Source: Australian Bureau of Statistics.']
Total rows: 711


Council,Year,Population*,_c3,*Source: Australian Bureau of Statistics.
Alpine Shire Council,2016-17,12638,null,null
Alpine Shire Council,2017-18,12730,null,null
Alpine Shire Council,2018-19,12812,null,null
Alpine Shire Council,2019-20,12973,null,null
Alpine Shire Council,2020-21,13012,null,null
Alpine Shire Council,2021-22,13069,null,null
Alpine Shire Council,2022-23,13175,null,null
Alpine Shire Council,2023-24,13182,null,null
Alpine Shire Council,2024-25,13199,null,null
Ararat Rural City Council,2016-17,11756,null,null


Year,Rows,Councils
2016-17,79,79
2017-18,79,79
2018-19,79,79
2019-20,79,79
2020-21,79,79
2021-22,79,79
2022-23,79,79
2023-24,79,79
2024-25,79,79


In [0]:

from pyspark.sql import functions as F

required_years = [
    "2020-21", "2021-22", "2022-23",
    "2023-24", "2024-25"
]

population_2020_25 = population.filter(
    F.col("Year").isin(required_years)
)

print("Filtered rows:", population_2020_25.count())

display(
    population_2020_25.groupBy("Year")
    .agg(
        F.count("*").alias("Rows"),
        F.countDistinct("Council").alias("Councils")
    )
    .orderBy("Year")
)


Filtered rows: 395


Year,Rows,Councils
2020-21,79,79
2021-22,79,79
2022-23,79,79
2023-24,79,79
2024-25,79,79


In [0]:

from pyspark.sql import functions as F

p = population_2020_25

# Check coverage
print("Rows:", p.count())
print("Councils:", p.select("Council").distinct().count())
print("Years:", p.select("Year").distinct().count())

# Check duplicate council-year combinations
duplicates = (
    p.groupBy("Council", "Year")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate council-year keys:", duplicates.count())

# Check missing and non-numeric population values
population_value = F.trim(F.col("Population*"))

invalid_population = p.filter(
    population_value.isNull()
    | (population_value == "")
    | (
        F.try_to_number(
            F.regexp_replace(population_value, ",", ""),
            F.lit("999999999999.99")
        ).isNull()
    )
)

print("Missing/invalid population:", invalid_population.count())

# Show reporting-year coverage
display(
    p.groupBy("Year")
    .agg(
        F.count("*").alias("Rows"),
        F.countDistinct("Council").alias("Councils")
    )
    .orderBy("Year")
)


Rows: 395
Councils: 79
Years: 5
Duplicate council-year keys: 0
Missing/invalid population: 0


Year,Rows,Councils
2020-21,79,79
2021-22,79,79
2022-23,79,79
2023-24,79,79
2024-25,79,79



## Results and Outcomes

### Source Schema Assessment
- Compared column names across available CSV sources.
- Generated a summary of source file structures.
- Identified the ABS population CSV as requiring
  additional header preparation.

### Population Validation
- Original Population.csv records: 711
- Selected reporting period: 2020–21 to 2024–25
- Filtered records: 395
- Distinct councils: 79
- Financial years: 5
- Duplicate council-year combinations: 0
- Missing or invalid population values: 0

### Business and Reporting Implications
- The selected population data has the expected
  council-year coverage.
- Population figures can support per-capita
  financial reporting after council-name matching
  and population definition checks.
- Financial data requires separate record-grain
  and numeric validation before integration.

### Output
A documented source schema and population
readiness assessment.

No source data was modified or overwritten.

### Next Steps
- Notebook 04: Validate council names and categories.
- Notebook 05: Clean performance data into Silver.
- Notebook 06: Validate and integrate financial
  and population datasets.
